# 04 · Training

The trained Gaussians on the frames kept out of training, next to the frames themselves. Run `avatar-splats train` first; this notebook only reads its results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from avatar_splats.body_models.hybrid import Avatar
from avatar_splats.config import load_config
from avatar_splats.fitting import fitted_path
from avatar_splats.gaussians import BoundGaussians, GaussianRecord
from avatar_splats.training import Evaluation, load_views, render_view, target, trained_path
from avatar_splats.training.views import head_gaussians

config = load_config("../configs/default.yaml")
device = config.training.device
avatar = Avatar.load(fitted_path(config, "avatar"))
gaussians = BoundGaussians(GaussianRecord.load(trained_path(config, "gaussians"))).to(device)
evaluation = Evaluation.load(trained_path(config, "evaluation"))
views = load_views(config, avatar)
faces = torch.from_numpy(avatar.faces).to(device)
head = head_gaussians(avatar, gaussians.faces)
print(f"{len(gaussians)} Gaussians")

## Quality on held-out frames

PSNR in dB within the compared region (above the chin for the head videos).

In [ ]:
for video in np.unique(evaluation.videos):
    chosen = evaluation.videos == video
    plt.plot(
        evaluation.frames[chosen],
        evaluation.psnr[chosen],
        "o-",
        label=f"{video}: {evaluation.psnr[chosen].mean():.1f} dB",
    )
plt.xlabel("frame")
plt.ylabel("PSNR (dB)")
plt.legend();

## Renders

Top: the frame with the background removed. Bottom: the avatar rendered for it. Pick a video and frames below.

In [ ]:
video = "body"
chosen = [v for v in views if v.video == video and not v.frame % config.training.holdout_every][:5]
black = torch.zeros(3, device=device)
fig, axes = plt.subplots(2, len(chosen), figsize=(4 * len(chosen), 10))
with torch.no_grad():
    for column, view in enumerate(chosen):
        image, _ = render_view(gaussians, faces, view, head, config.training.sh_degree, black)
        axes[0, column].imshow(target(view, black).cpu().numpy())
        axes[1, column].imshow(image.cpu().numpy())
        axes[0, column].set_title(f"frame {view.frame}")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()